# <font color="#418FDE" size="6.5" uppercase>**Density Clustering**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply DBSCAN, HDBSCAN, and OPTICS to discover density-based clusters. 
- Tune eps, min_samples, minimum cluster size, and distance metrics. 
- Use BIRCH and clustering metrics for larger or structured clustering tasks. 


## **1. DBSCAN Family**

### **1.1. DBSCAN Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_A/image_01_01.jpg?v=1784022494" width="250">



>* Finds clusters as dense connected regions
>* Handles irregular shapes and isolated noise

>* Core, border, and noise point roles
>* Clusters expand while outliers remain flagged

>* Expands clusters from dense connected neighborhoods
>* Finds noise without preselecting cluster count



In [ ]:
#@title Python Code - DBSCAN Basics

# Demonstrate DBSCAN on simple synthetic clusters.
# Show dense groups and noise points.
# Compare labels with one scatter plot.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

# Create curved data where centroid methods struggle.
points, true_groups = make_moons(
    n_samples=300,
    noise=0.07,
    random_state=42,
)

# Add a few isolated points to represent noise.
noise_points = np.array([[-1.5, 1.2], [2.2, -0.8], [2.6, 1.0]])
points = np.vstack([points, noise_points])

# Standardizing makes eps use comparable feature scales.
scaled_points = StandardScaler().fit_transform(points)

# DBSCAN groups dense neighborhoods and labels outliers as minus one.
dbscan = DBSCAN(eps=0.25, min_samples=5)
cluster_labels = dbscan.fit_predict(scaled_points)

# Count clusters while excluding the special noise label.
unique_labels = sorted(set(cluster_labels))
cluster_count = len([label for label in unique_labels if label != -1])
noise_count = int(np.sum(cluster_labels == -1))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"DBSCAN found {cluster_count} clusters.")
print(f"DBSCAN marked {noise_count} points as noise.")

# Plot clusters, using black crosses for noise points.
fig, ax = plt.subplots(figsize=(7, 5))
for label in unique_labels:
    mask = cluster_labels == label
    if label == -1:
        ax.scatter(points[mask, 0], points[mask, 1], c="black", marker="x", label="noise")
    else:
        ax.scatter(points[mask, 0], points[mask, 1], s=35, label=f"cluster {label}")

ax.set_title("DBSCAN finds dense curved clusters and noise")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend()
plt.show()



### **1.2. Epsilon and Min Samples**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_A/image_01_02.jpg?v=1784022496" width="250">



>* Epsilon and min samples define density
>* DBSCAN finds irregular clusters and noise

>* Epsilon sets DBSCAN’s neighborhood scale
>* Too small splits; too large merges

>* Min samples controls required cluster evidence
>* Tune with epsilon for stable results



In [ ]:
#@title Python Code - Epsilon and Min Samples

# This example tunes DBSCAN neighborhood settings.
# Epsilon and min_samples define local density.
# The plot shows clusters and noise points.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

# Make a small curved dataset for density clustering.
features, true_groups = make_moons(
    n_samples=300,
    noise=0.07,
    random_state=42,
)

# Standard scaling makes epsilon distances easier to compare.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Validate the simple two-feature shape before clustering.
if scaled_features.shape != (300, 2):
    raise ValueError("Expected 300 rows and 2 numeric features.")

# Fit DBSCAN with one epsilon and min_samples choice.
dbscan = DBSCAN(eps=0.25, min_samples=5)
cluster_labels = dbscan.fit_predict(scaled_features)

# Count clusters while excluding the noise label.
unique_labels = sorted(set(cluster_labels))
cluster_count = len([label for label in unique_labels if label != -1])
noise_count = int(np.sum(cluster_labels == -1))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"DBSCAN settings: eps=0.25, min_samples=5")
print(f"Clusters found: {cluster_count}; noise points: {noise_count}")

# Color noise separately from discovered dense clusters.
plot_colors = np.where(cluster_labels == -1, -1, cluster_labels)
fig, ax = plt.subplots(figsize=(7, 5))

scatter = ax.scatter(
    scaled_features[:, 0],
    scaled_features[:, 1],
    c=plot_colors,
    cmap="viridis",
    s=35,
)

ax.set_title("DBSCAN with epsilon 0.25 and min_samples 5")
ax.set_xlabel("Scaled feature 1")
ax.set_ylabel("Scaled feature 2")
plt.show()



### **1.3. Distance Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_A/image_01_03.jpg?v=1784022499" width="250">



>* Distance defines DBSCAN neighborhood density.
>* Scale features before choosing Euclidean distance.

>* Metrics define similarity for each problem
>* Match distance choice to data structure

>* Metrics guide HDBSCAN and OPTICS neighborhoods
>* Preprocessing improves meaningful density-based similarity



In [ ]:
#@title Python Code - Distance Metrics

# Compare distance metrics for DBSCAN neighborhoods.
# Scaling changes which points count as nearby.
# The plot shows metric-dependent cluster labels.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

# Create two compact groups with deterministic random variation.
points, true_groups = make_blobs(
    n_samples=160,
    centers=[[-2, 0], [2, 0]],
    cluster_std=[0.55, 0.55],
    random_state=42,
)

# Stretch one feature to make raw Euclidean distance misleading.
stretched_points = points.copy()
stretched_points[:, 1] = stretched_points[:, 1] * 8

# Standardization gives both features comparable influence.
scaled_points = StandardScaler().fit_transform(stretched_points)

# Fit DBSCAN twice, changing only preprocessing and metric meaning.
raw_labels = DBSCAN(eps=1.2, min_samples=5, metric="euclidean").fit_predict(
    stretched_points
)
scaled_labels = DBSCAN(eps=0.45, min_samples=5, metric="euclidean").fit_predict(
    scaled_points
)

# Count clusters while ignoring the DBSCAN noise label.
raw_cluster_count = len(set(raw_labels) - {-1})
scaled_cluster_count = len(set(scaled_labels) - {-1})
raw_noise_count = int(np.sum(raw_labels == -1))
scaled_noise_count = int(np.sum(scaled_labels == -1))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Raw stretched data: {raw_cluster_count} clusters, {raw_noise_count} noise points")
print(f"Scaled data: {scaled_cluster_count} clusters, {scaled_noise_count} noise points")

# Plot the scaled result to show the recovered density structure.
fig, ax = plt.subplots(figsize=(6, 4))
scatter = ax.scatter(
    scaled_points[:, 0],
    scaled_points[:, 1],
    c=scaled_labels,
    cmap="tab10",
    s=35,
)

ax.set_title("DBSCAN after scaling: Euclidean distance is meaningful")
ax.set_xlabel("Scaled feature 1")
ax.set_ylabel("Scaled feature 2")
ax.legend(*scatter.legend_elements(), title="Cluster label", loc="best")
plt.show()



## **2. Advanced Density Tuning**

### **2.1. HDBSCAN Tuning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_A/image_02_01.jpg?v=1784022501" width="250">



>* Find clusters across changing data densities
>* Tune for stable structure, not fixed distance

>* Set cluster size to match meaningful groups
>* Adjust min samples for density strictness

>* Choose metrics matching data type
>* Balance diagnostics, interpretation, and useful noise



In [ ]:
#@title Python Code - HDBSCAN Tuning

# This example tunes HDBSCAN on synthetic density clusters.
# Minimum cluster size changes noise and cluster counts.
# The plot shows one tuned clustering result.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.cluster import HDBSCAN
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Create clusters with different spreads and extra background noise.
cluster_points, true_labels = make_blobs(
    n_samples=[120, 90, 70],
    centers=[[-3, 0], [2, 2], [3, -2]],
    cluster_std=[0.25, 0.55, 0.85],
    random_state=42,
)

rng = np.random.default_rng(42)
noise_points = rng.uniform(low=-5, high=5, size=(35, 2))
features = np.vstack([cluster_points, noise_points])

# Scale features so distance comparisons are balanced.
scaled_features = StandardScaler().fit_transform(features)
if scaled_features.shape != (315, 2):
    raise ValueError("Expected 315 rows and 2 columns.")

# Try a few minimum cluster sizes while keeping min_samples fixed.
settings = [10, 25, 45]
summary_rows = []

for minimum_cluster_size in settings:
    model = HDBSCAN(
        min_cluster_size=minimum_cluster_size,
        min_samples=10,
        metric="euclidean",
    )
    labels = model.fit_predict(scaled_features)
    cluster_count = len(set(labels)) - int(-1 in labels)
    noise_percent = 100 * np.mean(labels == -1)
    clustered_mask = labels != -1
    score = np.nan
    if cluster_count > 1:
        score = silhouette_score(scaled_features[clustered_mask], labels[clustered_mask])
    summary_rows.append((minimum_cluster_size, cluster_count, noise_percent, score))

# Fit the selected setting for the final visualization.
chosen_model = HDBSCAN(
    min_cluster_size=25,
    min_samples=10,
    metric="euclidean",
)
chosen_labels = chosen_model.fit_predict(scaled_features)

print(f"scikit-learn version: {sklearn_version}")
print("min_cluster_size | clusters | noise % | silhouette")
for row in summary_rows:
    score_text = "n/a" if np.isnan(row[3]) else str(round(row[3], 2))
    print(f"{row[0]:16d} | {row[1]:8d} | {row[2]:7.1f} | {score_text}")

# Plot the chosen HDBSCAN labels, with noise shown separately.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    scaled_features[:, 0],
    scaled_features[:, 1],
    c=chosen_labels,
    cmap="tab10",
    s=35,
    edgecolor="black",
    linewidth=0.3,
)

ax.set_title("HDBSCAN tuning with min_cluster_size=25")
ax.set_xlabel("Scaled feature 1")
ax.set_ylabel("Scaled feature 2")
ax.legend(*scatter.legend_elements(), title="Cluster label", loc="best")
plt.show()



### **2.2. Minimum Cluster Size**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_A/image_02_02.jpg?v=1784022503" width="250">



>* Smaller values reveal finer cluster details
>* Larger values favor broader, stable groups

>* Balance statistics with domain knowledge
>* Test values for stable, useful clusters

>* Tune size with density and distance choices
>* Validate clusters for meaning and coherence



In [ ]:
#@title Python Code - Minimum Cluster Size

# This example tunes minimum cluster size.
# HDBSCAN reveals broader or smaller groups.
# The plot compares two beginner settings.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import HDBSCAN
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
import sklearn

# Create three uneven synthetic density clusters.
points, true_groups = make_blobs(
    n_samples=[35, 90, 160],
    centers=[[-3, 0], [0, 2], [3, 0]],
    cluster_std=[0.25, 0.45, 0.65],
    random_state=42,
)

# Add a few scattered points that may become noise.
noise = np.array([[-4, 3], [-2, -2], [1, -2], [4, 3], [5, -1]])
points = np.vstack([points, noise])

# Validate the simple two-feature clustering dataset.
if points.shape != (290, 2):
    raise ValueError("Expected 290 rows and 2 numeric features.")

# Fit HDBSCAN twice with different minimum cluster sizes.
small_model = HDBSCAN(min_cluster_size=15, min_samples=5)
large_model = HDBSCAN(min_cluster_size=70, min_samples=5)
small_labels = small_model.fit_predict(points)
large_labels = large_model.fit_predict(points)

# Count clusters while ignoring the noise label.
def count_clusters(labels):
    cluster_labels = set(labels)
    cluster_labels.discard(-1)
    return len(cluster_labels)

# Summarize how the setting changes detail and noise.
small_noise = int(np.sum(small_labels == -1))
large_noise = int(np.sum(large_labels == -1))
print("scikit-learn version:", sklearn.__version__)
print("min_cluster_size=15 clusters:", count_clusters(small_labels))
print("min_cluster_size=15 noise points:", small_noise)
print("min_cluster_size=70 clusters:", count_clusters(large_labels))
print("min_cluster_size=70 noise points:", large_noise)

# Compute silhouette only for non-noise clustered points.
clustered = large_labels != -1
large_score = silhouette_score(points[clustered], large_labels[clustered])
print("larger setting silhouette:", round(large_score, 3))

# Plot the more conservative clustering result.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    points[:, 0],
    points[:, 1],
    c=large_labels,
    cmap="tab10",
    s=45,
    edgecolor="black",
)

# Label the axes and title for interpretation.
ax.set_title("HDBSCAN with larger minimum cluster size")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend(*scatter.legend_elements(), title="Cluster", loc="best")
plt.show()



### **2.3. OPTICS Parameter Tuning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_A/image_02_03.jpg?v=1784022505" width="250">



>* OPTICS reveals clusters with varying densities
>* min_samples balances detail and stability

>* eps limits OPTICS neighborhood search scale
>* Choose eps based on analysis goals

>* Choose distance metrics suited to data type
>* Scale features and inspect reachability patterns



In [ ]:
#@title Python Code - OPTICS Parameter Tuning

# This example tunes OPTICS on synthetic clusters.
# Parameters change how dense neighborhoods are detected.
# The plot shows labels from one tuned setting.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import OPTICS
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

# Two blob groups create different local densities.
centers = [[-2.5, 0.0], [2.5, 0.0]]
cluster_std = [0.35, 1.0]

# The generated data is deterministic for repeatable teaching.
features, true_groups = make_blobs(
    n_samples=[120, 180], centers=centers, cluster_std=cluster_std, random_state=42
)

# Scaling keeps both coordinates equally important.
scaled_features = StandardScaler().fit_transform(features)

# Validate the small two-dimensional clustering dataset.
if scaled_features.shape != (300, 2):
    raise ValueError("Expected 300 rows and 2 features.")

# Compare a few beginner-friendly OPTICS settings.
settings = [
    {"min_samples": 5, "max_eps": 0.7, "metric": "euclidean"},
    {"min_samples": 20, "max_eps": 0.7, "metric": "euclidean"},
    {"min_samples": 20, "max_eps": 1.5, "metric": "euclidean"},
]

print("scikit-learn version:", sklearn.__version__)
print("Setting | clusters found | noise points")

# Count clusters and noise for each parameter choice.
for setting in settings:
    optics = OPTICS(
        min_samples=setting["min_samples"], max_eps=setting["max_eps"],
        metric=setting["metric"], cluster_method="xi"
    )
    labels = optics.fit_predict(scaled_features)
    cluster_count = len(set(labels) - {-1})
    noise_count = int(np.sum(labels == -1))
    label = "min_samples=" + str(setting["min_samples"])
    label = label + ", max_eps=" + str(setting["max_eps"])
    print(label, "|", cluster_count, "|", noise_count)

# Fit one tuned model for the visual result.
tuned_optics = OPTICS(
    min_samples=20, max_eps=1.5, metric="euclidean", cluster_method="xi"
)

# Labels use -1 for points treated as noise.
tuned_labels = tuned_optics.fit_predict(scaled_features)

# Plot the tuned OPTICS clustering result.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    scaled_features[:, 0], scaled_features[:, 1], c=tuned_labels, cmap="tab10", s=28
)

ax.set_title("OPTICS tuning with min_samples=20 and max_eps=1.5")
ax.set_xlabel("Scaled feature 1")
ax.set_ylabel("Scaled feature 2")
ax.legend(*scatter.legend_elements(), title="Cluster label", loc="best")
plt.show()



## **3. Large Scale Clustering**

### **3.1. Reachability Plot Insights**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_A/image_03_01.jpg?v=1784022507" width="250">



>* OPTICS plots reveal density-based cluster structure
>* Valleys show clusters; peaks mark boundaries

>* Valley shapes reveal cluster size and stability
>* Nested dips show density changes and subclusters

>* Guide parameters, cluster extraction, and noise handling
>* Summarize density patterns for real-world interpretation



In [ ]:
#@title Python Code - Reachability Plot Insights

# This example visualizes OPTICS reachability distances.
# Valleys suggest dense clusters at different scales.
# The plot helps diagnose large clustering structure.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import OPTICS
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
import sklearn

# Create uneven synthetic clusters for density exploration.
cluster_centers = [(-4, 0), (0, 0), (4, 0)]
cluster_spreads = [0.25, 0.65, 1.1]

features, true_groups = make_blobs(
    n_samples=[120, 160, 220],
    centers=cluster_centers,
    cluster_std=cluster_spreads,
    random_state=42,
)

# Add a few scattered points to mimic noise.
rng = np.random.default_rng(42)
noise = rng.uniform(low=-6, high=6, size=(35, 2))
features = np.vstack([features, noise])

# Standardization makes distances comparable across features.
scaled_features = StandardScaler().fit_transform(features)

if scaled_features.shape[0] < 50:
    raise ValueError("This example needs at least 50 observations.")

# OPTICS records reachability distances in its learned ordering.
optics = OPTICS(min_samples=12, xi=0.05, min_cluster_size=0.08)
optics.fit(scaled_features)

ordered_reachability = optics.reachability_[optics.ordering_]
finite_reachability = ordered_reachability[np.isfinite(ordered_reachability)]

# Replace the first infinite value for a readable plot scale.
plot_reachability = ordered_reachability.copy()
plot_reachability[~np.isfinite(plot_reachability)] = finite_reachability.max()

cluster_count = len(set(optics.labels_) - {-1})
noise_count = int(np.sum(optics.labels_ == -1))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"OPTICS clusters found: {cluster_count}")
print(f"Points labeled as noise: {noise_count}")

# Low valleys show dense regions in the OPTICS ordering.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_reachability, color="steelblue", linewidth=1.5)
ax.set_title("OPTICS Reachability Plot: Valleys Indicate Dense Clusters")
ax.set_xlabel("OPTICS ordering position")
ax.set_ylabel("Reachability distance")
ax.grid(True, alpha=0.3)
plt.show()



### **3.2. BIRCH Clustering**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_A/image_03_02.jpg?v=1784022509" width="250">



>* Summarizes massive datasets while scanning.
>* Builds compact trees for efficient clustering.

>* Threshold controls subcluster size and detail
>* Subclusters enable faster final grouping

>* Best for compact, well-scaled clusters
>* Summarizes large data for later evaluation



In [ ]:
#@title Python Code - BIRCH Clustering

# Demonstrate BIRCH on compact synthetic clusters.
# Compare threshold choices using clustering metrics.
# Expect clearer clusters with a suitable threshold.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import Birch
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Generate a small clustering dataset with known local groups.
features, true_groups = make_blobs(
    n_samples=900,
    centers=4,
    cluster_std=0.75,
    random_state=42,
)

# Scale features so distance-based clustering treats axes fairly.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Validate the data shape before fitting the clustering model.
if scaled_features.shape != (900, 2):
    raise ValueError("Expected 900 rows and 2 numeric features.")

# Fit BIRCH with a moderate threshold and four final clusters.
birch_model = Birch(threshold=0.45, n_clusters=4)
cluster_labels = birch_model.fit_predict(scaled_features)

# Count discovered clusters and evaluate separation with silhouette score.
cluster_count = len(set(cluster_labels))
silhouette = silhouette_score(scaled_features, cluster_labels)

# Fit a second BIRCH model to show threshold granularity.
fine_model = Birch(threshold=0.20, n_clusters=None)
fine_labels = fine_model.fit_predict(scaled_features)

# Print concise teaching results for the fitted models.
print("scikit-learn version:", sklearn.__version__)
print("Final BIRCH clusters:", cluster_count)
print("Silhouette score:", round(silhouette, 3))
print("Subclusters with smaller threshold:", len(set(fine_labels)))

# Plot the final BIRCH cluster assignments in feature space.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    scaled_features[:, 0],
    scaled_features[:, 1],
    c=cluster_labels,
    cmap="tab10",
    s=18,
)

# Label the plot so the clustering result is easy to read.
ax.set_title("BIRCH clustering after feature scaling")
ax.set_xlabel("Scaled feature 1")
ax.set_ylabel("Scaled feature 2")
ax.legend(*scatter.legend_elements(), title="Cluster")
plt.show()



### **3.3. Clustering Performance Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_A/image_03_03.jpg?v=1784022511" width="250">



>* Assess cluster usefulness and separation
>* Compare settings with domain context

>* Assess cluster structure without ground-truth labels
>* Balance metric scores with meaningful cluster shapes

>* Compare clusters with labels and outcomes
>* Balance metrics, scalability, stability, and usefulness



In [ ]:
#@title Python Code - Clustering Performance Metrics

# Compare clustering metrics on compact synthetic data.
# BIRCH summarizes data before assigning cluster labels.
# Scores show why one metric is insufficient.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import Birch
from sklearn.datasets import make_blobs
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import silhouette_score

# Create a small clustering dataset with known visual groups.
features, true_groups = make_blobs(
    n_samples=900,
    centers=4,
    cluster_std=[0.55, 0.75, 0.6, 0.8],
    random_state=42,
)

# Validate the basic shape before clustering.
if features.shape != (900, 2):
    raise ValueError("Expected 900 rows and 2 features.")

# Fit one scalable BIRCH model for large-scale clustering.
birch_model = Birch(n_clusters=4, threshold=0.7, branching_factor=50)
cluster_labels = birch_model.fit_predict(features)

# Count discovered clusters before computing internal metrics.
cluster_count = len(np.unique(cluster_labels))
if cluster_count < 2:
    raise ValueError("Metrics need at least two discovered clusters.")

# Compute three common internal clustering performance metrics.
silhouette = silhouette_score(features, cluster_labels)
calinski = calinski_harabasz_score(features, cluster_labels)
davies = davies_bouldin_score(features, cluster_labels)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"BIRCH discovered clusters: {cluster_count}")
print(f"Silhouette score, higher is better: {silhouette:.3f}")
print(f"Calinski-Harabasz score, higher is better: {calinski:.1f}")
print(f"Davies-Bouldin score, lower is better: {davies:.3f}")

# Plot the BIRCH labels to connect scores with structure.
fig, ax = plt.subplots(figsize=(6, 4))
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=cluster_labels,
    s=18,
    cmap="tab10",
)

ax.set_title("BIRCH clusters evaluated with internal metrics")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Density Clustering**</font>


In this lecture, you learned to:
- Apply DBSCAN, HDBSCAN, and OPTICS to discover density-based clusters. 
- Tune eps, min_samples, minimum cluster size, and distance metrics. 
- Use BIRCH and clustering metrics for larger or structured clustering tasks. 

In the next Lecture (Lecture B), we will go over 'Biclustering Validation'